# Forecaster Calibration Adjuster

This notebook experiments with adjusting a forecasting agents forecast based off of prior data of the forecast, with the
idea that the forecasters calibration can be improved.

This notebook assumed the following directory structure:
- `raw_forecast_data`: CSV files containing forecast resolutions.
- `generated`: Stored intermediary data objects.

## Data Filtering

This notebook filters and joins the raw forecasting data files. This notebook only focuses on one forecaster at a time,
so the forecaster is selected and filtered for before joining. Additionally, unresolved and nullified forecasts are also removed.

In [7]:
import pandas as pd
from os import mkdir, listdir
from os.path import isdir

# Names that the bot goes by in the data files.
#
# The bot may go under different names in the different files, so this is an array.
forecaster_names = ["mf-bot-1", "metac-gpt-4o"]

raw_df = pd.DataFrame()

# Collect data for the specified forecaster.
for file in listdir("raw_forecast_data"):
    with open(f"raw_forecast_data/{file}", "r") as f:
        df = pd.read_csv(f, sep=",", header=0)
        filtered_df = df[df["forecaster"].isin(forecaster_names)]
        raw_df = pd.concat([raw_df, filtered_df])

# Sort forecasts by when they were placed.
raw_df = raw_df.sort_values("forecast_timestamp")
# Remove forecasts that are not assessable.
raw_df = raw_df[~raw_df["resolution"].isin(["annulled", "ambiguous"])]

# Save dataframe.

if not isdir("generated"):
    mkdir("generated")
raw_df.to_csv("generated/filtered.csv", index=False)

raw_df.head()

,forecast_id,question_id,post_id,question_title,created_at,author_id,probability_yes,probability_yes_per_category,continuous_cdf,forecast_timestamp,...,cp_reveal_time,type,options,range_min,range_max,open_lower_bound,open_upper_bound,zero_point,project_title,forecast_endtime
33113,27711345,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.165007+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33112,27711346,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.388122+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33111,27711347,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.581658+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33110,27711348,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.784028+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33109,27711349,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.975425+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN


## Data Reshaping

The data needs to be converted into simple yes/no forecasts for the processing that is going to be done. There are multiple types of questions (enumerated in the below cell), each needs its own strategy.

In [8]:
raw_df["type"].drop_duplicates()

33113            numeric
33107    multiple_choice
33100             binary
Name: type, dtype: object

### Binary questions

In [17]:
raw_binary = raw_df[raw_df["type"] == "binary"]
raw_binary

,forecast_id,question_id,post_id,question_title,created_at,author_id,probability_yes,probability_yes_per_category,continuous_cdf,forecast_timestamp,...,cp_reveal_time,type,options,range_min,range_max,open_lower_bound,open_upper_bound,zero_point,project_title,forecast_endtime
33100,27711358,31264,31732,Will the bubble in the Magnificent Seven pop b...,2025-01-17 19:02:43.940868+00,236038,0.30,NaN,NaN,2025-01-20 02:45:25.938045+00,...,2025-01-20 03:27:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33099,27711359,31264,31732,Will the bubble in the Magnificent Seven pop b...,2025-01-17 19:02:43.940868+00,236038,0.30,NaN,NaN,2025-01-20 02:45:26.115718+00,...,2025-01-20 03:27:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33098,27711360,31264,31732,Will the bubble in the Magnificent Seven pop b...,2025-01-17 19:02:43.940868+00,236038,0.30,NaN,NaN,2025-01-20 02:45:26.291206+00,...,2025-01-20 03:27:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33097,27711361,31264,31732,Will the bubble in the Magnificent Seven pop b...,2025-01-17 19:02:43.940868+00,236038,0.25,NaN,NaN,2025-01-20 02:45:26.499799+00,...,2025-01-20 03:27:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33096,27711362,31264,31732,Will the bubble in the Magnificent Seven pop b...,2025-01-17 19:02:43.940868+00,236038,0.30,NaN,NaN,2025-01-20 02:45:26.727713+00,...,2025-01-20 03:27:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,28055187,35621,36184,"Will Elon Musk, Donald Trump, or JD Vance visi...",2025-03-15 15:49:28.723389+00,236038,0.40,NaN,NaN,2025-03-20 16:08:34.628939+00,...,2025-03-20 18:00:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
220,28055188,35621,36184,"Will Elon Musk, Donald Trump, or JD Vance visi...",2025-03-15 15:49:28.723389+00,236038,0.30,NaN,NaN,2025-03-20 16:08:34.87578+00,...,2025-03-20 18:00:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
219,28055189,35621,36184,"Will Elon Musk, Donald Trump, or JD Vance visi...",2025-03-15 15:49:28.723389+00,236038,0.30,NaN,NaN,2025-03-20 16:08:35.111318+00,...,2025-03-20 18:00:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
218,28055190,35621,36184,"Will Elon Musk, Donald Trump, or JD Vance visi...",2025-03-15 15:49:28.723389+00,236038,0.25,NaN,NaN,2025-03-20 16:08:35.393851+00,...,2025-03-20 18:00:00+00,binary,NaN,NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN



Binary questions are already in the desired format, and just need to be reshaped to the new table structure. The dataframe initialized and previewed in the following code block holds the structure that the other forecast type classes must adhere to.

In [34]:
reshaped_df = pd.DataFrame()
reshaped_binary = pd.DataFrame()
reshaped_binary["prediction"] = raw_binary["probability_yes"]
reshaped_binary["resolution"] = raw_binary["resolution"] == "yes"
reshaped_df = pd.concat([reshaped_df, reshaped_binary])
reshaped_binary

,prediction,resolution
33100,0.30,False
33099,0.30,False
33098,0.30,False
33097,0.25,False
33096,0.30,False
...,...,...
221,0.40,False
220,0.30,False
219,0.30,False
218,0.25,False


### Multiple choice questions

In [19]:
raw_mc = raw_df[raw_df["type"] == "multiple_choice"]
raw_mc

,forecast_id,question_id,post_id,question_title,created_at,author_id,probability_yes,probability_yes_per_category,continuous_cdf,forecast_timestamp,...,cp_reveal_time,type,options,range_min,range_max,open_lower_bound,open_upper_bound,zero_point,project_title,forecast_endtime
33107,27711351,31262,31730,"For Q1 2025, how many banks will be listed on ...",2025-01-17 19:02:43.857529+00,236038,NaN,"[0.1,0.5,0.25,0.1,0.05]",NaN,2025-01-20 02:45:18.652083+00,...,2025-01-20 03:27:00+00,multiple_choice,"[""0"",""1"",""2-3"",""4-6"","">6""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33106,27711352,31262,31730,"For Q1 2025, how many banks will be listed on ...",2025-01-17 19:02:43.857529+00,236038,NaN,"[0.1,0.5,0.25,0.1,0.05]",NaN,2025-01-20 02:45:18.855392+00,...,2025-01-20 03:27:00+00,multiple_choice,"[""0"",""1"",""2-3"",""4-6"","">6""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33105,27711353,31262,31730,"For Q1 2025, how many banks will be listed on ...",2025-01-17 19:02:43.857529+00,236038,NaN,"[0.1,0.5,0.25,0.1,0.05]",NaN,2025-01-20 02:45:19.06202+00,...,2025-01-20 03:27:00+00,multiple_choice,"[""0"",""1"",""2-3"",""4-6"","">6""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33104,27711354,31262,31730,"For Q1 2025, how many banks will be listed on ...",2025-01-17 19:02:43.857529+00,236038,NaN,"[0.05,0.6,0.25,0.08,0.02]",NaN,2025-01-20 02:45:19.246118+00,...,2025-01-20 03:27:00+00,multiple_choice,"[""0"",""1"",""2-3"",""4-6"","">6""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33103,27711355,31262,31730,"For Q1 2025, how many banks will be listed on ...",2025-01-17 19:02:43.857529+00,236038,NaN,"[0.10000000000000002,0.6000000000000001,0.2000...",NaN,2025-01-20 02:45:19.4416+00,...,2025-01-20 03:27:00+00,multiple_choice,"[""0"",""1"",""2-3"",""4-6"","">6""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,28055741,35705,36264,Which podcast will be ranked higher on Spotify...,2025-03-20 19:35:15.771896+00,236038,NaN,"[0.7,0.3]",NaN,2025-03-20 19:49:36.210767+00,...,2025-03-20 20:00:00+00,multiple_choice,"[""Call Her Daddy"",""Candace""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
111,28055742,35705,36264,Which podcast will be ranked higher on Spotify...,2025-03-20 19:35:15.771896+00,236038,NaN,"[0.7,0.3]",NaN,2025-03-20 19:49:36.563162+00,...,2025-03-20 20:00:00+00,multiple_choice,"[""Call Her Daddy"",""Candace""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
110,28055743,35705,36264,Which podcast will be ranked higher on Spotify...,2025-03-20 19:35:15.771896+00,236038,NaN,"[0.7,0.3]",NaN,2025-03-20 19:49:36.915573+00,...,2025-03-20 20:00:00+00,multiple_choice,"[""Call Her Daddy"",""Candace""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
109,28055744,35705,36264,Which podcast will be ranked higher on Spotify...,2025-03-20 19:35:15.771896+00,236038,NaN,"[0.7,0.3]",NaN,2025-03-20 19:49:37.267514+00,...,2025-03-20 20:00:00+00,multiple_choice,"[""Call Her Daddy"",""Candace""]",NaN,NaN,False,False,NaN,Q1 AI Forecasting Benchmark Tournament,NaN


Multiple choice questions are fairly simple to break down, every choice can be considered an independent question. Thus, each question just needs to be split into several individual sub-questions.

In [35]:
import ast

reshaped_mc = pd.DataFrame()
for idx, row in raw_mc.iterrows():
    predictions = ast.literal_eval(row["probability_yes_per_category"])
    options = ast.literal_eval(row["options"])

    resolution = row["resolution"]
    true_idx = options.index(resolution)
    resolution = [False] * len(options)
    resolution[true_idx] = True

    mc_append_df = pd.DataFrame()
    mc_append_df["prediction"] = predictions
    mc_append_df["resolution"] = resolution
    reshaped_mc = pd.concat([reshaped_mc, mc_append_df])

reshaped_df = pd.concat([reshaped_df, reshaped_mc])
reshaped_mc

,prediction,resolution
0,0.10,True
1,0.50,False
2,0.25,False
3,0.10,False
4,0.05,False
...,...,...
1,0.30,True
0,0.70,False
1,0.30,True
0,0.70,False


### Numeric questions

In [13]:
raw_numeric = raw_df[raw_df["type"] == "numeric"]
raw_numeric

,forecast_id,question_id,post_id,question_title,created_at,author_id,probability_yes,probability_yes_per_category,continuous_cdf,forecast_timestamp,...,cp_reveal_time,type,options,range_min,range_max,open_lower_bound,open_upper_bound,zero_point,project_title,forecast_endtime
33113,27711345,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.165007+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33112,27711346,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.388122+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33111,27711347,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.581658+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33110,27711348,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.784028+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
33109,27711349,31263,31731,What percentage of the vote will Alexander Luk...,2025-01-17 19:02:43.897918+00,236038,NaN,NaN,"[0.05,0.0506666667,0.0513333333,0.052,0.052666...",2025-01-20 02:45:17.975425+00,...,2025-01-20 03:27:00+00,numeric,NaN,60.0,100.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1956,28049185,35600,36163,"How many subscribers will the ""datascience"" su...",2025-03-15 15:49:27.169354+00,236038,NaN,NaN,"[0.9212221667,0.9213660558,0.921509945,0.92165...",2025-03-18 22:24:25.779379+00,...,2025-03-19 00:00:00+00,numeric,NaN,2627333.0,2800000.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
1955,28049186,35600,36163,"How many subscribers will the ""datascience"" su...",2025-03-15 15:49:27.169354+00,236038,NaN,NaN,"[0.928416625,0.9285245419,0.9286324588,0.92874...",2025-03-18 22:24:26.001555+00,...,2025-03-19 00:00:00+00,numeric,NaN,2627333.0,2800000.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
1954,28049187,35600,36163,"How many subscribers will the ""datascience"" su...",2025-03-15 15:49:27.169354+00,236038,NaN,NaN,"[0.9212221667,0.9213660558,0.921509945,0.92165...",2025-03-18 22:24:26.195669+00,...,2025-03-19 00:00:00+00,numeric,NaN,2627333.0,2800000.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN
1942,28049734,35601,36164,"How many subscribers will the ""goodnews"" subre...",2025-03-15 15:49:27.201271+00,236038,NaN,NaN,"[0.05,0.0501,0.0502,0.0503,0.0504,0.0505,0.050...",2025-03-19 00:29:10.023916+00,...,2025-03-19 02:00:00+00,numeric,NaN,100000.0,120000.0,True,True,NaN,Q1 AI Forecasting Benchmark Tournament,NaN


In [36]:
reshaped_df

,prediction,resolution
33100,0.30,False
33099,0.30,False
33098,0.30,False
33097,0.25,False
33096,0.30,False
...,...,...
1,0.30,True
0,0.70,False
1,0.30,True
0,0.70,False
